In [0]:
from pyspark.sql import functions as F
from datetime import datetime
from delta import DeltaTable

#### Bronze layer ingestion
- get_last_successful_watermark() - reads the last processed watermark from the control table
- upsert_bronze_control() - updatest the control table after a successful Bronze Load

In [0]:
def get_last_successfull_watermark(table_name: str):
    ctrl = (
        spark.table("novacart_catalog.bronze_schema.ingestion_control")
        .filter(
            (F.col("layer") == "bronze") &
            (F.col("table_name") == table_name) &
            (F.col("run_status") == "success")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )
    rows = ctrl.collect()
    if not rows:
        return None, None
    last_successful_ts = rows[0]["last_successful_ts"]
    last_successful_pk = rows[0]["last_successful_pk"]
    return last_successful_ts, last_successful_pk

In [0]:
def upsert_bronze_control(table_name,ts_col,pk_col,last_ts,last_pk,rows_written,run_id):
   control_df = spark.createDataFrame(
       [(
           "bronze",
           table_name,
           ts_col,
           pk_col,
           last_ts,
           int(last_pk) if last_pk is not None else None,
           run_id,
           int(rows_written),
           "success",
           datetime.utcnow()
       )],
       schema = """
       layer string,
       table_name string,
       ts_col string,
       pk_col string,
       last_successful_ts timestamp,
       last_successful_pk bigint,
       last_run_id string,
       rows_written bigint,
       run_status string,
       updated_at timestamp
       """
   )
   dt = DeltaTable.forName(spark, "novacart_catalog.bronze_schema.ingestion_control")
   (
       dt.alias("t")
       .merge(
           control_df.alias("s"),
          F.expr("t.table_name = s.table_name and t.layer = s.layer")
       )
       .whenMatchedUpdate(
           set={
               "ts_col" : F.col("s.ts_col"),
               "pk_col" : F.col("s.pk_col"),
               "last_successful_ts" : F.col("s.last_successful_ts"),
               "last_successful_pk" : F.col("s.last_successful_pk"),
               "last_run_id" : F.col("s.last_run_id"),
               "rows_written" : F.col("s.rows_written"),
               "run_status" : F.col("s.run_status"),
               "updated_at" : F.col("s.updated_at")
           }
                                   
       )
       .whenNotMatchedInsertAll()
       .execute()
   )

#### Silver Layer Ingestion 
This cell contains reusable logic for Silver
- upsert_to_silver() - merges cleaned / transformed rows into the Silver target table
- get_last_processed_bronze_ingested_at() - reads the silver watermark
- upsert_silver_control() - updates the silver control table
- get_incremental_bronze() - reads only new Bronze rows that Silver has not processed yet

In [0]:
def upsert_to_silver(df_source, target_table, join_keys):
    if spark.catalog.tableExists(target_table):
        dynamic_join_condition = " AND ".join([f"source.{c} = target.{c}" for c in join_keys])
        print(f"Merge condition for {target_table} is {dynamic_join_condition}")
        dt = DeltaTable.forName(spark, target_table)
        (
            dt.alias("target")
            .merge(
                df_source.alias("source"),
                F.expr(dynamic_join_condition)
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_bronze_ingested_at(entity_name: str):
    ctrl = (
    spark.table("novacart_catalog.silver_schema.processing_control")
    .filter(
        (F.col("layer") == "silver") &
        (F.col("entity_name") == entity_name) &
        (F.col("run_status") == "success")
    )
    .orderBy(F.col("updated_at").desc())
    .limit(1)
    )

    rows = ctrl.collect()

    if not rows:
        return None

    return rows[0]["last_processed_bronze_ingested_at"]

In [0]:
def upsert_silver_control(entity_name, last_processed_bronze_run_id, last_processed_bronze_ingested_at, rows_merged, silver_run_id):
    control_df = spark.createDataFrame(
        [(
            "silver",
            entity_name,
            last_processed_bronze_run_id,
            last_processed_bronze_ingested_at,
            int(rows_merged),
            "success",
            datetime.utcnow(),
            silver_run_id            
        )],
        schema = """
        layer string,
        entity_name string,
        last_processed_bronze_run_id string,
        last_processed_bronze_ingested_at timestamp,
        rows_merged bigint,
        run_status string,       
        updated_at timestamp,
        silver_run_id string
        """
    )
    dt = DeltaTable.forName(spark, "novacart_catalog.silver_schema.processing_control")
    (
        dt.alias("t")
        .merge(
            control_df.alias("s"),
            F.expr("t.entity_name = s.entity_name and t.layer = s.layer")
        )
        .whenMatchedUpdate(
            set={
                "last_processed_bronze_run_id" : F.col("s.last_processed_bronze_run_id"),
                "last_processed_bronze_ingested_at" : F.col("s.last_processed_bronze_ingested_at"),
                "rows_merged" : F.col("s.rows_merged"),
                "run_status" : F.col("s.run_status"),
                "updated_at" : F.col("s.updated_at"),
                "silver_run_id": F.col("s.silver_run_id")
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
def get_incremental_bronze(bronze_table, entity_name):
    last_ingested_at = get_last_processed_bronze_ingested_at(entity_name)
    bronze_df = spark.read.table(bronze_table)

    if last_ingested_at is None:
        return bronze_df, last_ingested_at

    bronze_latest_records_df = bronze_df.filter(F.col("bronze_ingested_at") > last_ingested_at)

    return bronze_latest_records_df, last_ingested_at
